<a href="https://colab.research.google.com/github/ThierryHdaSilva/Agente_De_Conversa_DIO/blob/main/ConversaAgente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
language = 'pt'


# GRAVACAO AUDIO


In [ ]:
from IPython.display import Javascript, Audio, display
from google.colab import output
from base64 import b64decode
from io import BytesIO
import time
import os

!pip -q install pydub
from pydub import AudioSegment


RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))

const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  const chunks = []

  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()

  await sleep(time)

  recorder.onstop = async () => {
    const blob = new Blob(chunks)
    const text = await b2text(blob)
    stream.getTracks().forEach(track => track.stop())
    resolve(text)
  }

  recorder.stop()
})
"""


def gravar_audio(segundos=10):
    """Grava áudio do microfone e retorna um AudioSegment (pydub)."""
    display(Javascript(RECORD_JS))
    resultado_js = output.eval_js(f'record({segundos * 1000})')
    audio_bytes = b64decode(resultado_js.split(',')[1])
    audio = AudioSegment.from_file(BytesIO(audio_bytes))
    return audio


def salvar_audio(audio, prefixo='gravacao'):
    """Salva o AudioSegment em disco com nome único e retorna o caminho do arquivo."""
    nome_arquivo = f'{prefixo}_{int(time.time())}.wav'
    audio.export(nome_arquivo, format='wav')
    return nome_arquivo


def limpar_gravacoes_antigas(prefixo='gravacao'):
    """Remove arquivos .wav antigos gerados por essa sessão, para não acumular espaço."""
    for f in os.listdir('.'):
        if f.startswith(prefixo) and f.endswith('.wav'):
            os.remove(f)


# --- Uso ---
limpar_gravacoes_antigas()

print('Ouvindo...\n')
audio = gravar_audio(segundos=10)
arquivo_audio = salvar_audio(audio, prefixo='gravacao')

print(f'Duração: {len(audio)} ms')
print(f'Volume: {audio.dBFS:.2f} dBFS')
display(Audio(arquivo_audio, autoplay=False))

# RECONHECIMENTO DE FALA

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q

In [ ]:
import whisper

# Carrega o modelo (roda 1x só, reutiliza depois)
if 'model' not in globals():
    model = whisper.load_model('small')

resultado = model.transcribe(arquivo_audio, language='pt')
texto = resultado['text']

print('Transcrição:', texto)

# API

In [ ]:
!pip -q install google-genai

In [ ]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

In [ ]:
def perguntar_gemini(pergunta, modelo='gemini-2.5-flash'):
    resposta = client.models.generate_content(
        model=modelo,
        contents=pergunta
    )
    return resposta.text


resposta_texto = perguntar_gemini(texto)
print('Você disse:', texto)
print('\nGemini respondeu:', resposta_texto)

# VOZ SINTETIZADA

In [ ]:
!pip -q install gTTS

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display
from pydub import AudioSegment
from pydub.effects import speedup
import time

def falar(texto_resposta, idioma='pt', velocidade=1.3):
    """Converte texto em áudio (TTS), acelera e retorna o caminho do arquivo final."""
    # Gera o áudio original (voz normal)
    tts = gTTS(text=texto_resposta, lang=idioma)
    nome_temp = f'temp_{int(time.time())}.mp3'
    tts.save(nome_temp)

    # Acelera o áudio
    audio = AudioSegment.from_file(nome_temp)
    audio_acelerado = speedup(audio, playback_speed=velocidade)

    nome_final = f'resposta_{int(time.time())}.mp3'
    audio_acelerado.export(nome_final, format='mp3')

    return nome_final


# --- Uso ---
arquivo_resposta = falar(resposta_texto, velocidade=1.3)
display(Audio(arquivo_resposta, autoplay=True))